In [ ]:
from syft_rds.orchestra import setup_rds_stack
from rds_chat_analysis import REPO_ROOT
import pandas as pd
import dotenv
from langchain.chat_models import init_chat_model
import os
from rds_chat_analysis import DATA_DIR

In [ ]:
key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds",
    key=key,
    log_level="DEBUG",
    reset=False,
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
wildchat_dataset = ds_client.datasets[0]
wildchat_dataset.describe()

In [ ]:
mock_data = pd.read_parquet(
    wildchat_dataset.mock_path / "data.parquet",
)

mock_data

# Load LLM

In [ ]:
dotenv.load_dotenv(wildchat_dataset.mock_path / "credentials.env")

llm = init_chat_model(
    model=os.environ["OPENROUTER_MODEL_NAME"],
    model_provider=os.environ["MODEL_PROVIDER"],
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=os.environ["OPENROUTER_API_URL"],
)

# Extract facets


In [ ]:
from rds_chat_analysis.facets import extract_facets

facets_df = extract_facets(
    conversation_df=mock_data,
    llm=llm,
    llm_cache_dir=DATA_DIR / "Wildchat-10k" / "mock",
)

In [ ]:
facets_df

In [ ]:
# llm.invoke("Test message, please respond with 'Hello World!'")

In [ ]:
from rds_chat_analysis.facet_prompts import (
    format_facet_extraction_prompt,
    FACET_EXTRACTORS,
)

print("Available facet extractors:")
for key in FACET_EXTRACTORS.keys():
    print(f"- {key}")


# Get a random conversation from the mock data
conversation = mock_data.iloc[10]["conversation"]

# Use the "Request" facet extractor to format the prompt
facet_extractor_kwargs = FACET_EXTRACTORS["Request"]
messages = format_facet_extraction_prompt(
    conversation=conversation,
    **facet_extractor_kwargs,
)

In [ ]:
from rich.console import Console
from langchain_core.messages import BaseMessage
import textwrap


def display_messages(messages: list[BaseMessage], width: int = 120):
    console = Console(highlight=False, soft_wrap=True)
    for msg in messages:
        role = msg.type.capitalize()
        wrapped = "\n".join(
            textwrap.fill(line, width=width)
            for line in msg.content.strip().splitlines()
        )
        console.print(f"[bold green]{role.upper()}:[/bold green]\n{wrapped}")


display_messages(messages)

In [ ]:
result = llm.invoke(messages)


def extract_answer(text: str) -> str | None:
    text = text.strip()
    if "<answer>" in text:
        text = text.split("<answer>", 1)[-1]
    if "</answer>" in text:
        text = text.split("</answer>", 1)[0]
    return text.strip() or None


answer = extract_answer(result.content)
print(f"Extracted answer: {answer}")

In [ ]:
result.model_dump(mode="json")

In [ ]:
import json
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from langchain.chat_models.base import BaseChatModel
from rds_chat_analysis.facet_prompts import (
    format_facet_extraction_prompt,
    FACET_EXTRACTORS,
)
from rds_chat_analysis import DATA_DIR


def extract_facets(
    conversation_df: pd.DataFrame,
    output_path: str | Path,
    facet_extractor_kwargs: dict,
    llm: BaseChatModel,
    rps: int = 50,
    num_retries: int = 3,
) -> None:
    retrying_llm = llm.with_retry(stop_after_attempt=num_retries)

    output_path = Path(output_path)
    seen = set()

    if output_path.exists():
        with output_path.open() as f:
            for line in f:
                seen.add(json.loads(line)["id"])

    delay = 1.0 / rps

    with output_path.open("a") as out:
        for row in tqdm(conversation_df.itertuples(), total=len(conversation_df)):
            if row.id in seen:
                continue

            messages = format_facet_extraction_prompt(
                conversation=list(row.conversation),
                **facet_extractor_kwargs,
            )

            try:
                response = retrying_llm.invoke(messages)
            except Exception as e:
                print(f"Error processing row {row.id}: {e}")
                continue

            response_json = response.model_dump(mode="json")
            json.dump({"id": row.id, "response": response_json}, out)
            out.write("\n")
            time.sleep(delay)

In [ ]:
facet_name = "Request"
output_path = DATA_DIR / "Wildchat-10k" / f"facet_{facet_name.lower()}_mock.jsonl"
facet_kwargs = FACET_EXTRACTORS[facet_name]

extract_facets(
    conversation_df=mock_data,
    output_path=output_path,
    facet_extractor_kwargs=facet_kwargs,
    llm=llm,
)

In [ ]:
request_facet_results = pd.read_json(
    output_path,
    lines=True,
)

# Extract the answers and put in a new column
request_facet_results["request"] = request_facet_results["response"].apply(
    lambda x: extract_answer(x["content"])
)

facet_df = pd.DataFrame(request_facet_results[["id", "request"]])

In [ ]:
from lingua import LanguageDetectorBuilder

lingua_detector = LanguageDetectorBuilder.from_all_languages().build()


def extract_language(
    conversation_df: pd.DataFrame,
) -> dict[str, str]:
    lingua_detector = LanguageDetectorBuilder.from_all_languages().build()
    languages = {}

    for row in conversation_df.itertuples():
        conversation_concatenated = "\n".join(
            [msg["content"] for msg in row.conversation]
        )
        language = lingua_detector.detect_language_of(conversation_concatenated)
        languages[row.id] = language.name if language else "unknown"
    return languages


language_results = extract_language(mock_data)

# add to facet_df as 'language'
facet_df["language"] = facet_df["id"].map(language_results)

In [ ]:
print(facet_df.head())

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings


cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

# We use gte-multilingual, best mix of fast, small, high MTEB scores
MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"

model_kwargs = {
    "device": "cuda:0" if cuda_available else "cpu",
    "trust_remote_code": True,  # Required to run gte-multilingual in sentence-transformers
}
encode_kwargs = {"normalize_embeddings": False}
embedder = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)


store = LocalFileStore(DATA_DIR / "Wildchat-10k" / "embedding_cache")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedder,
    store,
    namespace=MODEL_NAME,
)

In [ ]:
import umap
import umap.plot
import numpy as np
import pandas as pd
import hdbscan
from sklearn.metrics.pairwise import cosine_distances

request_embeddings = np.array(facet_df["request_embedding"].tolist())

# We use UMAP to R20 to reduce the dimensionality of the embeddings before clustering
# This is a common trick to improve clustering performance
umap_for_clusterer = umap.UMAP(
    n_components=20,
    n_neighbors=15,
    min_dist=0.01,
    metric="cosine",
    random_state=42,
)
embeddings_for_clusterer = umap_for_clusterer.fit_transform(request_embeddings)

# Precompute cosine distances for HDBSCAN
dists = cosine_distances(embeddings_for_clusterer).astype("float64")
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=10,
    metric="precomputed",
    cluster_selection_method="eom",
)
assignments = hdbscan_model.fit_predict(dists)
facet_df["request_cluster"] = assignments

In [ ]:
umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

embedding_2d = umap_model.fit_transform(request_embeddings)

facet_df["embedding_2d"] = list(embedding_2d)

In [ ]:
facet_df

In [ ]:
from rds_chat_analysis.cluster_prompts import (
    format_cluster_description_prompt,
    FACET_CRITERIA,
)

clusters = facet_df["request_cluster"].unique()

cluster_description_responses = {}
for cluster_id in clusters:
    if cluster_id == -1:
        continue

    in_df = facet_df[facet_df["request_cluster"] == cluster_id]
    out_df = facet_df[facet_df["request_cluster"] != cluster_id]

    in_sample = in_df["request"].sample(n=min(len(in_df), 50), random_state=42).tolist()
    out_sample = (
        out_df["request"].sample(n=min(len(out_df), 50), random_state=42).tolist()
    )

    messages = format_cluster_description_prompt(
        answers=in_sample,
        contrastive_answers=out_sample,
        criteria=FACET_CRITERIA["Request"],
    )

    response = llm.invoke(messages).content
    cluster_description_responses[cluster_id] = response

In [ ]:
import re


def clean_description(text: str) -> dict[str, str]:
    if not text.startswith("<summary>"):
        text = "<summary>" + text
    summary = re.search(r"<summary>(.*?)</summary>", text, re.DOTALL)
    name = re.search(r"<name>(.*?)</name>", text, re.DOTALL)

    return {
        "name": name.group(1).strip() if name else "",
        "summary": summary.group(1).strip() if summary else "",
    }


cluster_descriptions = {
    -1: {
        "name": "Unassigned",
        "summary": "This cluster contains requests that do not fit into any other cluster.",
    },
}

for id_, response in cluster_description_responses.items():
    cleaned = clean_description(response)
    cluster_descriptions[int(id_)] = cleaned

In [ ]:
publishable_fields = [
    "language",
    "embedding_2d",
    "request_cluster",
    "cluster_name",
    "cluster_summary",
]

publishable_df = facet_df[publishable_fields].copy()
print(publishable_df.head())